# Rubik AI: training the large model on Google Colab (GPU)

This notebook trains the **large CubeNet** (14.7 million parameters, an architecture comparable to the
DeepCubeA/EfficientCube papers) on Colab's free GPU, then tests it on the same 100 benchmark cubes as the laptop experiments.

**Run the cells from top to bottom.** All checkpoints are saved to Google Drive, so if Colab
disconnects (which happens often on the free tier), simply run all cells again: training
resumes automatically from the last checkpoint.

## 1. Make sure the GPU is on

Menu **Runtime → Change runtime type → T4 GPU → Save**. The cell below should print a table that mentions "Tesla T4".

In [ ]:
!nvidia-smi

## 2. Connect Google Drive

Checkpoints are saved to the folder `MyDrive/Rubik_AI_colab`. Allow access when asked.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/Rubik_AI_colab"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Storage folder:", DRIVE_DIR)

## 3. Get the code from GitHub

If the repository is **private**, cloning needs an **access token**. Create one once on GitHub:

1. Profile picture → **Settings** → **Developer settings** → **Personal access tokens** → **Fine-grained tokens** → **Generate new token**.
2. Give it a name (e.g. `colab-rubik`) and an expiry date (e.g. 30 days).
3. **Repository access → Only select repositories → Rubik_AI**.
4. **Permissions → Repository permissions → Contents → Read-only**.
5. Click **Generate token** and copy it.

The token is only typed into the box that appears when the cell runs. It is **not displayed, not saved
in the notebook, and removed from the Git configuration** once the code is downloaded.
For a public repository, just leave the box empty.

In [ ]:
import getpass, os, subprocess

REPO = "HaiqalGITuru/Rubik_AI"
WORKDIR = "/content/Rubik_AI"

if not os.path.exists(WORKDIR):
    token = getpass.getpass("Paste your GitHub token (leave empty for a public repository): ")
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    r = subprocess.run(["git", "clone", url, WORKDIR], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr)
    subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"])
    del token
else:
    print("Code already present in", WORKDIR)

os.chdir(WORKDIR)
!ls

## 4. Training settings

- `STEPS = 100_000` with 1,000 scrambles per batch (26,000 positions per step).
  Estimate on a T4: about 0.1–0.2 seconds per step, so **several hours in total**, most likely
  spread over several Colab sessions.
- **Do not change `STEPS` after training has started**, because the (cosine) learning-rate schedule is based on it.

In [ ]:
MODEL_PATH = f"{DRIVE_DIR}/cubenet_large.pt"
STEPS = 100_000
SCRAMBLES = 1000
print("Checkpoint:", MODEL_PATH, "| exists:", os.path.exists(MODEL_PATH))

## 5. Training

Speed (steps/s) and the estimated time left are printed every 200 steps. A checkpoint is saved to Drive
every 2,000 steps. To stop, click the **stop** button on this cell: the model is saved immediately.

If Colab disconnects: run cells 2, 3 and 4 again, then this cell. Training resumes automatically.

In [ ]:
!python -m scripts.train --device cuda --hidden1 5000 --hidden2 1000 --blocks 4 --scrambles {SCRAMBLES} --steps {STEPS} --cosine --log-every 200 --eval-every 2000 --save {MODEL_PATH} --resume

## 6. Training curves

Can be run at any time, including before training has finished (after stopping it).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(MODEL_PATH.replace(".pt", "_log.csv"))
ev = log.dropna(subset=["solve_d20"])

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].plot(log.step, log.loss, color="#2453E8"); ax[0].set_title("Loss"); ax[0].set_xlabel("step")
ax[1].plot(log.step, log.accuracy, color="#2453E8"); ax[1].set_title("Accuracy"); ax[1].set_xlabel("step")
for d, c in [(10, "#A9B3C2"), (15, "#6B85E8"), (20, "#2453E8")]:
    ax[2].plot(ev.step, ev[f"solve_d{d}"] * 100, "o-", color=c, ms=3, label=f"depth {d}")
ax[2].set_title("Success rate (quick evaluation)"); ax[2].set_ylabel("%"); ax[2].set_xlabel("step"); ax[2].legend(frameon=False)
for a in ax: a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{DRIVE_DIR}/training_curves_large.png", dpi=150)
plt.show()

## 7. Export a lightweight model

The training checkpoint is about 175 MB (too big for GitHub). This version only holds the model weights
in float16, about **29 MB**, with the same predictions. This is the file to download to your laptop.

In [ ]:
INFER_PATH = f"{DRIVE_DIR}/cubenet_large_infer.pt"
!python -m scripts.export_model {MODEL_PATH} {INFER_PATH}

## 8. Beam experiment on the same 100 benchmark cubes

Results are saved to Drive. **Note for the report:** times here are measured on a GPU, so they cannot be compared
directly with laptop times. What can be compared fairly is the **success rate and the solution length**.
The optional cell below tests the small (laptop) model on the same GPU, so the time comparison is fair too.

In [ ]:
import shutil
from IPython.display import Image, display

!python -m scripts.beam_experiment --model {INFER_PATH} --device cuda --name beam_experiment_large

shutil.copytree("results/beam_experiment_large", f"{DRIVE_DIR}/beam_experiment_large", dirs_exist_ok=True)
display(Image("results/beam_experiment_large/overview.png"))

In [ ]:
# Optional: the small model (trained on the laptop) tested on the same GPU, for a fair time comparison
!python -m scripts.beam_experiment --model models/cubenet.pt --device cuda --name beam_experiment_small_gpu
shutil.copytree("results/beam_experiment_small_gpu", f"{DRIVE_DIR}/beam_experiment_small_gpu", dirs_exist_ok=True)

## 9. Bring the results to your laptop

Open Google Drive in a browser, folder **Rubik_AI_colab**, and download:

- `cubenet_large_infer.pt` → save to `Rubik_AI/models/`
- `cubenet_large_log.csv` and `training_curves_large.png` → save to `Rubik_AI/results/colab/`
- the folder `beam_experiment_large` (and `beam_experiment_small_gpu` if you ran it) → save to `Rubik_AI/results/`

The file `cubenet_large.pt` (175 MB) does **not** need to be downloaded; keep it on Drive to resume training later if needed.